# Prepare 91-feature relative position and velocity forecasts

Run notebook **32_continued_1v1.ipynb** to completion first. This notebook ingests its two-perspective NN-duel corpus and prepares supervised data for forecasting the opposing aircraft relative to the observer at **+1, +2 and +5 seconds**. It prepares data only; it does not collect flights or train a model.

Every input frame contains exactly the **91 policy features used by the neural pilots in notebooks 30 and 31**: 47 compact aircraft/friendly-missile values, 40 hostile-missile values and four energy values. Nothing is appended: no acceleration, selected skill, pilot identity, outcome or future state. Histories contain the pilots' chronological **20 frames at 10 Hz**.

Each physical target is `[opponent_position - observer_position, opponent_velocity - observer_velocity]` in world North/West/Up at the future timestamp. `targets.npy` stores that six-vector in metres and metres/second. `residuals.npy` subtracts a constant-relative-velocity baseline, and the matching baseline and per-horizon masks are saved separately. This makes the prepared data usable for direct regression or stable residual regression without regenerating the corpus.

In [1]:
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_RELATIVE_MOTION_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((path.resolve() for path in candidates
             if (path / "bvr_behavior_prediction/forecasting/relative_motion.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_RELATIVE_MOTION_ROOT to the Trajectory_Classification checkout.")
sys.path.insert(0, str(ROOT))

from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the active kernel and native-library repair guidance.
import numpy as np
import pandas as pd
from IPython.display import display
from bvr_behavior_prediction.forecasting import relative_motion as motion
from bvr_behavior_prediction.forecasting.relative_motion import (
    HORIZONS, HISTORY_STEPS, RelativeMotionData, analyze_corpus, discover_corpus,
    nonoverlap_audit, prepare_dataset, read_json, sha256, write_json,
)

EXPERIMENT_ROOT = ROOT / "artifacts/relative_motion_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
source_default = ROOT / "shared/corpora/nn-duels-varied-v1/vw3"
source_setting = os.environ.get("BVR_RELATIVE_MOTION_CORPUS", str(source_default))
SOURCE = discover_corpus(ROOT, source_setting)
SOURCE_DIGEST = sha256(SOURCE / "manifest.json")
PREPARATION_DIGEST = sha256(motion.__file__)
DATA_DEFAULT = (EXPERIMENT_ROOT / "data" /
                f"{SOURCE.name}_{SOURCE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}")
DATA = Path(os.environ.get("BVR_RELATIVE_MOTION_DATA", str(DATA_DEFAULT))).expanduser().resolve()
SPLIT_SEED = 1729
print({"source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST,
       "prepared_output": str(DATA), "features_per_frame": 91,
       "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS.tolist()})

{'source': 'C:\\Users\\theon\\Trajectory_Classification\\shared\\corpora\\nn-duels-varied-v1\\vw3', 'source_manifest_sha256': '145be1e59986135f7c0a97c5903447a094f59f54aace6b87a005c0c4c7a17310', 'prepared_output': 'C:\\Users\\theon\\Trajectory_Classification\\artifacts\\relative_motion_forecasting\\data\\vw3_145be1e59986_688b3c4b', 'features_per_frame': 91, 'history_steps': 20, 'horizons_s': [1.0, 2.0, 5.0]}


## Alignment, split and leakage contract

Both aircraft perspectives of a physical flight remain in the same partition through `scenario_group`. The seeded split is 40% train, 30% test and 30% validation, stratified by initial-range quantiles. Training histories are thinned onto adjacent two-second slots and alternate A01/B01, so no raw 20-frame input interval overlaps another retained training interval on the physical-flight timeline. Test and validation retain dense one-second origins.

A history and its label are eligible only while both aircraft are alive. Each horizon is masked independently when its exact future frame is missing or either aircraft is dead. Labels come only from the physical `tracks` table; target decisions are never read. Input feature scaling and residual-target scaling are fitted only on retained training histories/labels. The raw 91-channel frame array remains unchanged and memory-mapped.

The loader checks corpus hashes, source roles, timestamp cadence, the compact observation geometry against physical tracks, split isolation, target/baseline consistency and cross-perspective training non-overlap. Choose a new data directory whenever the source manifest, preparation code or split seed changes.

In [2]:
analysis = analyze_corpus(SOURCE)
episodes = analysis["episodes"]
eligible = analysis["eligible_by_horizon"]
print(f"Surveyed {len(episodes):,} perspectives from {episodes.episode_id.nunique():,} physical flights.")
display(episodes.groupby("observer_id").agg(
    perspectives=("episode_id", "size"), physical_flights=("episode_id", "nunique"),
    mean_frames=("frames", "mean")))
display(pd.DataFrame({"horizon_s": HORIZONS,
                      "eligible_dense_labels": eligible.sum(axis=0)}))

ValueError: Expected notebook-32 NN duels sampled at 10 Hz

In [ ]:
if (DATA / "manifest.json").exists():
    manifest = read_json(DATA / "manifest.json")
    if manifest["source_manifest_sha256"] != SOURCE_DIGEST:
        raise ValueError("Existing data belongs to another corpus; choose a new output path.")
    if manifest["source_code_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Preparation code changed; choose a new output path.")
    if manifest["split_seed"] != SPLIT_SEED:
        raise ValueError("Split seed changed; choose a new output path.")
    print("Reusing verified prepared data:", DATA)
else:
    manifest = prepare_dataset(ROOT, DATA, SOURCE, split_seed=SPLIT_SEED, analysis=analysis)

data = RelativeMotionData(DATA)
assert data.features.shape[1] == 91
assert data.manifest["features"] == analysis["features"]
display(pd.DataFrame(manifest["counts"]).T)
display(pd.Series({"raw_frames": len(data.features), "features_per_frame": 91,
                   "feature_array_GiB": data.features.nbytes / 2**30,
                   "prepared_windows": len(data.samples)}))
display(pd.Series(nonoverlap_audit(data.samples)))
display(pd.read_csv(DATA / "target_support_by_split.csv"))

## Inspect an exact training example

The example below shows the source feature order, a raw unstandardised policy history and the three future physical targets. `target = constant-relative-velocity baseline + residual` is checked at every valid horizon. The final receipt is written only after all integrity checks pass.

In [ ]:
schema = read_json(SOURCE / "schema.json")
display(pd.DataFrame(schema["features"]))
sample_id = int(data.indices["train"][0])
sample = data.samples.iloc[sample_id]
history = data.history([sample_id], standardize=False)[0]
target = data.targets[sample_id]
baseline = data.baseline[sample_id]
residual = data.residuals[sample_id]
valid = data.valid[sample_id]
np.testing.assert_allclose(target[valid], (baseline + residual)[valid], rtol=1e-5, atol=1e-3)
display(sample)
display(pd.DataFrame({
    "history_step": np.arange(int(sample.step) - HISTORY_STEPS + 1, int(sample.step) + 1),
    "relative_n_m": history[:, 9] * 50000,
    "relative_w_m": history[:, 10] * 50000,
    "relative_u_m": history[:, 11] * 10000,
    "relative_vn_mps": history[:, 12] * 600,
    "relative_vw_mps": history[:, 13] * 600,
    "relative_vu_mps": history[:, 14] * 600,
}))
rows = []
for index, horizon in enumerate(HORIZONS):
    rows.append({"horizon_s": float(horizon), "valid": bool(valid[index]),
                 **dict(zip(data.manifest["target_fields"], target[index])),
                 **{f"residual_{name}": value for name, value in
                    zip(data.manifest["target_fields"], residual[index])}})
display(pd.DataFrame(rows))

write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST,
})
print("Relative-motion training data is ready:", DATA)